In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.stats import gaussian_kde

In [ ]:
info = pd.read_csv('full_sample_info.csv',index_col=0)
info

In [ ]:
data = pd.read_csv('../large_files/tabulated_counts_vhh.csv',index_col=0)
data.head()

In [ ]:
data.columns.tolist()

In [ ]:
data['cDNA_tot'] = data[[col for col in data.columns if 'rep' in col]].sum(axis=1)

In [ ]:
for col in data.columns:
    if 'tot' in col:
        data[col.replace('tot','freq')] = data[col]/data[col].sum()

In [ ]:
plt.figure(figsize=[13,4.3])
letters = 'ABC'

for a,animal in enumerate(info['animal'].unique()):
    plt.subplot(1,3,1+a)

    usable_data = data[data[[f'{animal}_rep1_freq',f'{animal}_rep2_freq']].max(axis=1)>0]

    low_rep1 = usable_data.loc[usable_data[f'{animal}_rep1_freq']>0,f'{animal}_rep1_freq'].min()
    low_rep2 = usable_data.loc[usable_data[f'{animal}_rep2_freq']>0,f'{animal}_rep2_freq'].min()
    low = min(low_rep1,low_rep2)/1.5
    kde = gaussian_kde(np.log10(usable_data[[f'{animal}_rep1_freq',f'{animal}_rep2_freq']].values.T + low))
    kde_out = np.log10(kde(np.log10(usable_data[[f'{animal}_rep1_freq',f'{animal}_rep2_freq']].values.T + low))+1e-3)
    print(low, np.min(kde_out), np.max(kde_out))
    
    ax = sns.scatterplot(data=usable_data,x=f'{animal}_rep1_freq',y=f'{animal}_rep2_freq',hue=kde_out)
    plt.plot([-1,1],[-1,1],'k--')

    if a==0:
        plt.xlabel('Rep. 1 Abundance',fontsize=16,loc='left')
        ax.annotate("", xytext=(0.7, -0.14), xy=(2.2, -0.14), xycoords='axes fraction',
                    arrowprops={'width':1.7,'color':'k'})
        
        plt.ylabel('Rep. 2 Abundance',fontsize=16)
    else:
        plt.xlabel('')
        plt.ylabel('')

    rep1_min = data[f'{animal}_rep1_freq'][data[f'{animal}_rep1_freq']>0].min()
    rep2_min = data[f'{animal}_rep2_freq'][data[f'{animal}_rep2_freq']>0].min()
    ax.set_xscale('symlog',linthresh=rep1_min,linscale=0.4)
    ax.set_yscale('symlog',linthresh=rep2_min,linscale=0.4)
    plt.xlim([-rep1_min,0.1])
    plt.ylim([-rep2_min,0.1])
    ax.text(-0.15, 1.05, letters[a], fontsize=32, transform=ax.transAxes)
    ax.text(0.05, 0.9, animal, fontsize=20, transform=ax.transAxes)
    
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.legend([],frameon=False)

plt.subplot(1,3,3)

low_cDNA = data.loc[data['cDNA_freq']>0,'cDNA_freq'].min()
low_phage = data.loc[data['phage_freq']>0,'phage_freq'].min()
low = min(low_cDNA,low_phage)/2
kde = gaussian_kde(np.log10(data[['cDNA_freq','phage_freq']].values.T + low))
kde_out = np.log10(kde(np.log10(data[['cDNA_freq','phage_freq']].values.T + low))+1e-3)
print(low, np.min(kde_out), np.max(kde_out))

ax = sns.scatterplot(data=data,x='cDNA_freq',y='phage_freq',hue=kde_out)
plt.plot([-1,1],[-1,1],'k--')

plt.xlabel('cDNA Abundance',fontsize=16)
plt.ylabel('Phage Abundance',fontsize=16)

cdna_min = data['cDNA_freq'][data['cDNA_freq']>0].min()
phage_min = data['phage_freq'][data['phage_freq']>0].min()
ax.set_xscale('symlog',linthresh=cdna_min,linscale=0.4)
ax.set_yscale('symlog',linthresh=phage_min,linscale=0.4)
plt.xlim([-cdna_min,0.1])
plt.ylim([-phage_min,0.1])
ax.text(-0.3, 1.05, letters[2], fontsize=32, transform=ax.transAxes)

plt.xticks([0,1e-6,1e-5,1e-4,1e-3,1e-2,1e-1],fontsize=12)
plt.yticks(fontsize=12)
plt.legend([],frameon=False)

pos = ax.get_position()
ax.set_position([pos.x0+0.05, pos.y0, pos.width, pos.height])

plt.savefig('figS14.png',dpi=300,bbox_inches='tight')